In [1]:
from pathlib import Path
import os

# Get current working directory
current_dir = Path(os.getcwd()).parent

# Adjust path based on your notebook location relative to data
npz_path_gc = current_dir / 'data' / '2-Data' / 'GoldCoast' / 'current_wind_20100101_20241231_GoaldCoast.npz'
stings_path_gc = current_dir / 'data'  / '2-Data' / 'GoldCoast' / 'goaldcoast_stings.csv'

npz_path_nc = current_dir / 'data' / '2-Data' / 'Sydney_Newcastle' / 'current_wind_20100101_20241231_Sydney.npz'
stings_path_nc = current_dir / 'data'  / '2-Data' / 'Sydney_Newcastle' / 'Sydney_stings.csv'


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load the stings data
stings_df = pd.read_csv(stings_path_nc)

# Convert time column to datetime
stings_df['time'] = pd.to_datetime(stings_df['time'], dayfirst=True)

# Sort by date to ensure proper order
stings_df = stings_df.sort_values('time').reset_index(drop=True)

# Calculate gaps between consecutive dates
stings_df['gap_days'] = stings_df['time'].diff().dt.days

# Remove the first row (which will have NaN gap)
gaps = stings_df['gap_days'].dropna()

# Get all gaps greater than 1 day (actual missing data)
missing_gaps = gaps[gaps > 1]

print(f"Total number of records: {len(stings_df)}")
print(f"Date range: {stings_df['time'].min()} to {stings_df['time'].max()}")
print(f"Total number of gaps > 1 day: {len(missing_gaps)}")
print(f"\n{'='*60}")

# Count frequency of each gap size
gap_counts = missing_gaps.value_counts().sort_index(ascending=False)

print("\nGap sizes in DESCENDING order with their frequencies:")
print(f"{'='*60}")
print(f"{'Gap Size (days)':<20} {'Count':<10} {'Total Missing Days':<20}")
print(f"{'-'*60}")

for gap_size, count in gap_counts.items():
    # Each gap of size N means N-1 missing days
    missing_days = (gap_size - 1) * count
    print(f"{int(gap_size):<20} {count:<10} {int(missing_days):<20}")

print(f"\n{'='*60}")
print(f"\nSummary Statistics:")
print(f"  Largest gap: {int(missing_gaps.max())} days")
print(f"  Smallest gap: {int(missing_gaps.min())} days")
print(f"  Average gap size: {missing_gaps.mean():.2f} days")
print(f"  Median gap size: {missing_gaps.median():.2f} days")

# Calculate total missing days
total_missing = sum((gap - 1) for gap in missing_gaps)
print(f"  Total missing days: {int(total_missing)}")

# Optional: Show details of the top 10 largest gaps with their date ranges
print(f"\n{'='*60}")
print("\nTop 10 Largest Gaps (with date ranges):")
print(f"{'='*60}")

# Find indices where gaps occur
gap_indices = stings_df[stings_df['gap_days'] > 1].index

# Create a list of gap details
gap_details = []
for idx in gap_indices:
    gap_size = int(stings_df.loc[idx, 'gap_days'])
    date_before = stings_df.loc[idx-1, 'time']
    date_after = stings_df.loc[idx, 'time']
    gap_details.append({
        'gap_size': gap_size,
        'from': date_before,
        'to': date_after
    })

# Sort by gap size and show top 10
gap_details_df = pd.DataFrame(gap_details)
gap_details_df = gap_details_df.sort_values('gap_size', ascending=False).head(10)

for i, row in gap_details_df.iterrows():
    print(f"\n{row['gap_size']} days: {row['from'].date()} → {row['to'].date()}")


Total number of records: 1821
Date range: 2009-01-01 00:00:00 to 2025-02-23 00:00:00
Total number of gaps > 1 day: 670


Gap sizes in DESCENDING order with their frequencies:
Gap Size (days)      Count      Total Missing Days  
------------------------------------------------------------
168                  1          167                 
159                  1          158                 
153                  1          152                 
152                  1          151                 
151                  1          150                 
147                  2          292                 
146                  2          290                 
139                  1          138                 
134                  1          133                 
133                  3          396                 
130                  1          129                 
125                  1          124                 
27                   1          26                  
20                   1

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def create_sequential_dataset(stings_csv_path, env_npz_path, n_days, output_path='sequential_dataset.npz'):
    """
    Create a sequential dataset where each sample contains n days of environmental data
    with the target being the sting value on the final day.
    
    Parameters:
    -----------
    stings_csv_path : str
        Path to the stings CSV file
    env_npz_path : str
        Path to the environmental NPZ file
    n_days : int
        Number of days to include in each sequence
    output_path : str
        Path to save the output NPZ file
        
    Returns:
    --------
    dict : Information about the created dataset
    """
    
    # Load stings data
    print(f"Loading stings data from {stings_csv_path}...")
    stings_df = pd.read_csv(stings_csv_path)
    stings_df['time'] = pd.to_datetime(stings_df['time'], dayfirst=True)
    stings_df = stings_df.sort_values('time').reset_index(drop=True)
    
    # normalize column name lookup (case-insensitive)
    col_map = {col.lower(): col for col in stings_df.columns}
    sum_col = col_map.get('stings_sum')
    binary_col = col_map.get('stings_binary')
    if sum_col is None or binary_col is None:
        raise KeyError(f"Expected columns 'stings_sum' and 'stings_binary' (case-insensitive). Found: {list(stings_df.columns)}")
    
    # Load environmental data
    print(f"Loading environmental data from {env_npz_path}...")
    env_data = np.load(env_npz_path)
    UVTempSalt_UVTs = env_data['UVTempSalt_UVTs']  # Shape: (5479, 6, 15, 15)
    
    # Environmental data start date
    env_start_date = datetime(2010, 1, 1)
    
    print(f"\nEnvironmental data shape: {UVTempSalt_UVTs.shape}")
    print(f"Environmental data covers: {env_start_date} to {env_start_date + timedelta(days=UVTempSalt_UVTs.shape[0]-1)}")
    print(f"Number of sting records: {len(stings_df)}")
    print(f"Sequence length (n_days): {n_days}")
    
    # Storage for the new dataset
    sequences = []
    targets_sum = []
    targets_binary = []
    valid_dates = []
    
    skipped_count = 0
    
    # Process each sting record
    for idx, row in stings_df.iterrows():
        sting_date = row['time']
        
        # Calculate the start date for the sequence (n_days before sting_date, inclusive)
        # If n_days=7 and sting_date is 7/1/2024, we want 1/1/2024 to 7/1/2024
        start_date = sting_date - timedelta(days=n_days - 1)
        
        # Calculate indices in the environmental data
        start_idx = (start_date - env_start_date).days
        end_idx = (sting_date - env_start_date).days
        
        # Check if we have enough data
        if start_idx < 0 or end_idx >= UVTempSalt_UVTs.shape[0]:
            skipped_count += 1
            continue
        
        # Check if we have exactly n_days
        if (end_idx - start_idx + 1) != n_days:
            skipped_count += 1
            continue
        
        # Extract the sequence (n_days of environmental data)
        sequence = UVTempSalt_UVTs[start_idx:end_idx+1, :, :, :]  # Shape: (n_days, 6, 15, 15)
        
        # Store the sequence and targets
        sequences.append(sequence)
        targets_sum.append(row[sum_col])
        targets_binary.append(row[binary_col])
        valid_dates.append(sting_date.strftime('%Y-%m-%d'))
        
        if len(sequences) % 100 == 0:
            print(f"Processed {len(sequences)} valid sequences...")
    
    # Convert to numpy arrays
    sequences = np.array(sequences)  # Shape: (num_samples, n_days, 6, 15, 15)
    targets_sum = np.array(targets_sum)
    targets_binary = np.array(targets_binary)
    valid_dates = np.array(valid_dates)
    
    print(f"\n{'='*60}")
    print(f"Dataset Creation Summary:")
    print(f"{'='*60}")
    print(f"Total sting records: {len(stings_df)}")
    print(f"Skipped records (insufficient data): {skipped_count}")
    print(f"Valid sequences created: {len(sequences)}")
    print(f"\nOutput data shapes:")
    print(f"  Sequences: {sequences.shape} (num_samples, n_days, channels, height, width)")
    print(f"  Targets (sum): {targets_sum.shape}")
    print(f"  Targets (binary): {targets_binary.shape}")
    print(f"  Dates: {valid_dates.shape}")
    
    # Save to NPZ file
    print(f"\nSaving dataset to {output_path}...")
    np.savez_compressed(
        output_path,
        sequences=sequences,
        targets_sum=targets_sum,
        targets_binary=targets_binary,
        dates=valid_dates,
        n_days=n_days,
        crop_lon_min=env_data['crop_lon_min'],
        crop_lon_max=env_data['crop_lon_max'],
        crop_lat_min=env_data['crop_lat_min'],
        crop_lat_max=env_data['crop_lat_max']
    )
    
    print(f"\n✓ Dataset saved successfully!")
    
    # Return summary information
    return {
        'total_records': len(stings_df),
        'skipped': skipped_count,
        'valid_samples': len(sequences),
        'sequence_shape': sequences.shape,
        'output_path': output_path
    }

# Example usage:
n_days = 7
processed_data_path_gc = current_dir / 'data' / 'processed' / 'GoldCoast' / f'GoldCoast_{n_days}_days.npz'
processed_data_path_nc = current_dir / 'data' / 'processed' / 'Sydney_Newcastle' / f'Sydney_Newcastle_{n_days}_days.npz'
result = create_sequential_dataset(
    stings_csv_path=stings_path_nc,
    env_npz_path=npz_path_nc,
    n_days=n_days,
    output_path=processed_data_path_nc
)


Loading stings data from c:\Users\VJ\Desktop\2025T3Group4\data\2-Data\Sydney_Newcastle\Sydney_stings.csv...
Loading environmental data from c:\Users\VJ\Desktop\2025T3Group4\data\2-Data\Sydney_Newcastle\current_wind_20100101_20241231_Sydney.npz...

Environmental data shape: (5479, 6, 13, 13)
Environmental data covers: 2010-01-01 00:00:00 to 2024-12-31 00:00:00
Number of sting records: 1821
Sequence length (n_days): 7
Processed 100 valid sequences...
Processed 200 valid sequences...
Processed 300 valid sequences...
Processed 400 valid sequences...
Processed 500 valid sequences...
Processed 600 valid sequences...
Processed 700 valid sequences...
Processed 800 valid sequences...
Processed 900 valid sequences...
Processed 1000 valid sequences...
Processed 1100 valid sequences...
Processed 1200 valid sequences...
Processed 1300 valid sequences...
Processed 1400 valid sequences...
Processed 1500 valid sequences...
Processed 1600 valid sequences...

Dataset Creation Summary:
Total sting record

In [4]:
import numpy as np

print("\nLoading sequential dataset from NPZ file...")
env_data = np.load(processed_data_path_nc)

# Load the sequential data
sequences = env_data['sequences']  # Shape: (num_samples, n_days, 6, 15, 15)
targets_sum = env_data['targets_sum']  # Shape: (num_samples,)
targets_binary = env_data['targets_binary']  # Shape: (num_samples,)
dates = env_data['dates']  # Shape: (num_samples,)
n_days = int(env_data['n_days'])  # Sequence length

# Geographic bounds
crop_lon_min = env_data['crop_lon_min']
crop_lon_max = env_data['crop_lon_max']
crop_lat_min = env_data['crop_lat_min']
crop_lat_max = env_data['crop_lat_max']

print(f"\nSequential dataset loaded successfully!")
print(f"{'='*60}")
print(f"Data shape: {sequences.shape}")
print(f"  - Number of samples: {sequences.shape[0]}")
print(f"  - Sequence length (n_days): {sequences.shape[1]} (also stored as: {n_days})")
print(f"  - Channels: {sequences.shape[2]}")
print(f"  - Spatial grid: {sequences.shape[3]}x{sequences.shape[4]}")
print(f"\nTarget data:")
print(f"  - Targets (sum) shape: {targets_sum.shape}")
print(f"  - Targets (binary) shape: {targets_binary.shape}")
print(f"  - Dates shape: {dates.shape}")
print(f"\nGeographic bounds:")
print(f"  - Longitude: [{crop_lon_min:.4f}, {crop_lon_max:.4f}]")
print(f"  - Latitude: [{crop_lat_min:.4f}, {crop_lat_max:.4f}]")
print(f"\nDate range:")
print(f"  - First date: {dates[0]}")
print(f"  - Last date: {dates[-1]}")
print(f"\nTarget statistics:")
print(f"  - Stings_sum range: [{targets_sum.min()}, {targets_sum.max()}]")
print(f"  - Stings_binary distribution: {np.bincount(targets_binary.astype(int))}")
print(f"    (0: no bluebottles, 1: bluebottles present)")



Loading sequential dataset from NPZ file...

Sequential dataset loaded successfully!
Data shape: (1653, 7, 6, 13, 13)
  - Number of samples: 1653
  - Sequence length (n_days): 7 (also stored as: 7)
  - Channels: 6
  - Spatial grid: 13x13

Target data:
  - Targets (sum) shape: (1653,)
  - Targets (binary) shape: (1653,)
  - Dates shape: (1653,)

Geographic bounds:
  - Longitude: [151.3410, 152.5790]
  - Latitude: [-34.0150, -32.8630]

Date range:
  - First date: 2010-01-07
  - Last date: 2024-12-30

Target statistics:
  - Stings_sum range: [0, 4450]
  - Stings_binary distribution: [1220  433]
    (0: no bluebottles, 1: bluebottles present)


In [5]:
"""
==============================================================================
3D CNN TRAINING PIPELINE FOR BLUEBOTTLE STING PREDICTION
Complete All-in-One Script - TEMPORAL SPLIT VERSION
==============================================================================

This script contains:
1. Data loading and preprocessing (NaN handling, normalization)
2. Three lightweight 3D CNN architectures
3. Training pipeline with validation (TEMPORAL SPLIT - NO RANDOMIZATION)
4. Comprehensive evaluation and visualization
5. Model comparison and analysis

IMPORTANT: Uses temporal splitting to preserve time series order
- Training set: Earliest data
- Validation set: Middle period
- Test set: Most recent data

Author: Your Name
Date: October 2025
==============================================================================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')




# Set random seeds for reproducibility (model initialization only)
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU'))} device(s)")
print("="*80 + "\n")



TensorFlow version: 2.10.1
GPU Available: 1 device(s)



In [6]:

# ==============================================================================
# SECTION 1: DATA LOADING AND PREPROCESSING
# ==============================================================================

def load_and_preprocess_data(npz_path, location_name='Unknown'):
    """
    Load data from NPZ file and handle NaN values
    
    Parameters:
    -----------
    npz_path : str or Path
        Path to NPZ file containing sequences, targets_binary, dates
    location_name : str
        Name of location for printing
        
    Returns:
    --------
    sequences_normalized : ndarray
        Preprocessed sequences (samples, n_days, channels, H, W)
    targets_binary : ndarray
        Binary targets (samples,)
    dates : ndarray
        Date strings (samples,)
    """
    print(f"\n{'='*80}")
    print(f"LOADING {location_name.upper()} DATA")
    print(f"{'='*80}")
    print(f"File: {npz_path}")
    
    data = np.load(npz_path)
    
    sequences = data['sequences']  # Shape: (samples, n_days, channels, H, W)
    targets_binary = data['targets_binary']
    dates = data['dates']
    
    print(f"\\nOriginal shape: {sequences.shape}")
    print(f"  - Samples: {sequences.shape[0]}")
    print(f"  - Days: {sequences.shape[1]}")
    print(f"  - Channels: {sequences.shape[2]}")
    print(f"  - Spatial: {sequences.shape[3]}x{sequences.shape[4]}")
    print(f"\\nTargets shape: {targets_binary.shape}")
    print(f"Date range: {dates[0]} to {dates[-1]}")
    
    # Handle NaN values
    print(f"\n{'─'*80}")
    print("HANDLING NaN VALUES")
    print(f"{'─'*80}")
    nan_count_before = np.isnan(sequences).sum()
    print(f"NaN count before: {nan_count_before:,} ({nan_count_before/sequences.size*100:.2f}%)")
    
    # Replace NaN with 0 (land mask remains as 0)
    sequences_clean = np.nan_to_num(sequences, nan=0.0)
    
    nan_count_after = np.isnan(sequences_clean).sum()
    print(f"NaN count after: {nan_count_after:,}")
    print("✓ NaN values replaced with 0 (land mask)")
    
    # Normalize data (channel-wise normalization)
    print(f"\n{'─'*80}")
    print("NORMALIZING DATA (Channel-wise)")
    print(f"{'─'*80}")
    
    n_samples, n_days, n_channels, h, w = sequences_clean.shape
    sequences_normalized = np.zeros_like(sequences_clean)
    
    channel_names = ['U Current', 'V Current', 'Temperature', 'Salinity', 'U Wind', 'V Wind']
    
    for ch in range(n_channels):
        channel_data = sequences_clean[:, :, ch, :, :]
        
        # Get non-zero values for mean/std calculation (exclude land)
        non_zero_mask = channel_data != 0
        if non_zero_mask.any():
            mean_val = channel_data[non_zero_mask].mean()
            std_val = channel_data[non_zero_mask].std()
            
            if std_val > 0:
                sequences_normalized[:, :, ch, :, :] = np.where(
                    non_zero_mask,
                    (channel_data - mean_val) / std_val,
                    0
                )
            else:
                sequences_normalized[:, :, ch, :, :] = channel_data
        else:
            sequences_normalized[:, :, ch, :, :] = channel_data
        
        ch_name = channel_names[ch] if ch < len(channel_names) else f'Channel {ch+1}'
        print(f"  ✓ {ch_name:15s} - Mean: {mean_val:8.3f}, Std: {std_val:8.3f}")
    
    # Class distribution
    print(f"\n{'─'*80}")
    print("CLASS DISTRIBUTION")
    print(f"{'─'*80}")
    unique, counts = np.unique(targets_binary, return_counts=True)
    for u, c in zip(unique, counts):
        label = 'No Stings (0)' if u == 0 else 'Stings (1)'
        print(f"  {label:15s}: {c:4d} samples ({c/len(targets_binary)*100:5.1f}%)")
    
    print(f"\\n✓ Data loading and preprocessing complete!")
    
    return sequences_normalized, targets_binary, dates


def temporal_train_val_test_split(X, y, dates, train_ratio=0.64, val_ratio=0.16, test_ratio=0.20):
    """
    Split data temporally (chronologically) without randomization
    
    Parameters:
    -----------
    X : ndarray
        Features
    y : ndarray
        Labels
    dates : ndarray
        Date strings
    train_ratio : float
        Proportion for training (default 0.64)
    val_ratio : float
        Proportion for validation (default 0.16)
    test_ratio : float
        Proportion for testing (default 0.20)
        
    Returns:
    --------
    X_train, X_val, X_test, y_train, y_val, y_test, dates_train, dates_val, dates_test
    """
    n_samples = len(X)
    
    # Calculate split indices
    train_end = int(n_samples * train_ratio)
    val_end = int(n_samples * (train_ratio + val_ratio))
    
    # Split chronologically (no shuffling)
    X_train = X[:train_end]
    y_train = y[:train_end]
    dates_train = dates[:train_end]
    
    X_val = X[train_end:val_end]
    y_val = y[train_end:val_end]
    dates_val = dates[train_end:val_end]
    
    X_test = X[val_end:]
    y_test = y[val_end:]
    dates_test = dates[val_end:]
    
    return X_train, X_val, X_test, y_train, y_val, y_test, dates_train, dates_val, dates_test



In [7]:

# ==============================================================================
# SECTION 2: MODEL ARCHITECTURES
# ==============================================================================

def create_model_1_simple_3dcnn(input_shape, name="Simple3DCNN"):
    """
    Model 1: Simple 3D CNN with minimal layers
    
    Architecture:
    - 2 Conv3D blocks (16 → 32 filters)
    - Global Average Pooling (handles any spatial size)
    - Single dense layer
    
    Parameters: ~50K
    Best for: Baseline, fast training, less overfitting
    """
    model = models.Sequential(name=name)
    
    # First 3D Conv block
    model.add(layers.Conv3D(16, kernel_size=(2, 3, 3), activation='relu', 
                            padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling3D(pool_size=(1, 2, 2)))
    model.add(layers.Dropout(0.2))
    
    # Second 3D Conv block
    model.add(layers.Conv3D(32, kernel_size=(2, 3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling3D(pool_size=(1, 2, 2)))
    model.add(layers.Dropout(0.3))
    
    # Adaptive pooling to handle different spatial resolutions (15x15 or 12x12)
    model.add(layers.GlobalAveragePooling3D())
    
    # Dense layers
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    return model


def create_model_2_c3d_lite(input_shape, name="C3D_Lite"):
    """
    Model 2: C3D-inspired lightweight architecture
    
    Architecture:
    - 3 Conv3D blocks with 3x3x3 kernels (C3D style)
    - Progressive filters: 16 → 32 → 32
    - Global Average Pooling
    - Two dense layers
    
    Parameters: ~150K
    Best for: Better feature extraction, more capacity
    """
    model = models.Sequential(name=name)
    
    # Conv block 1
    model.add(layers.Conv3D(16, kernel_size=(3, 3, 3), activation='relu',
                            padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling3D(pool_size=(1, 2, 2)))
    
    # Conv block 2
    model.add(layers.Conv3D(32, kernel_size=(3, 3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.25))
    
    # Conv block 3
    model.add(layers.Conv3D(32, kernel_size=(3, 3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling3D(pool_size=(2, 2, 2)))
    model.add(layers.Dropout(0.3))
    
    # Adaptive pooling
    model.add(layers.GlobalAveragePooling3D())
    
    # Dense layers
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    return model


def create_model_3_separable_3dcnn(input_shape, name="Separable3DCNN"):
    """
    Model 3: Separable 3D CNN with factorized convolutions
    
    Architecture:
    - Factorized (1,3,3) spatial + (3,1,1) temporal convolutions
    - Reduces parameters while maintaining capacity
    - 3 blocks: 16 → 32 → 64 filters
    - Global Average Pooling
    - Two dense layers
    
    Parameters: ~120K
    Best for: Parameter efficiency, separate spatial-temporal learning
    """
    inputs = layers.Input(shape=input_shape)
    
    # Block 1: Spatial then temporal convolution
    x = layers.Conv3D(16, kernel_size=(1, 3, 3), activation='relu', padding='same')(inputs)
    x = layers.Conv3D(16, kernel_size=(3, 1, 1), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)
    x = layers.Dropout(0.2)(x)
    
    # Block 2: Spatial then temporal convolution
    x = layers.Conv3D(32, kernel_size=(1, 3, 3), activation='relu', padding='same')(x)
    x = layers.Conv3D(32, kernel_size=(3, 1, 1), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 3
    x = layers.Conv3D(64, kernel_size=(1, 3, 3), activation='relu', padding='same')(x)
    x = layers.Conv3D(64, kernel_size=(2, 1, 1), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Adaptive pooling
    x = layers.GlobalAveragePooling3D()(x)
    
    # Dense layers
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name=name)
    
    return model


def print_model_summary(model):
    """Print detailed model architecture and parameter count"""
    print(f"\n{'='*80}")
    print(f"MODEL: {model.name}")
    print(f"{'='*80}")
    model.summary()
    trainable = sum([tf.size(w).numpy() for w in model.trainable_weights])
    non_trainable = sum([tf.size(w).numpy() for w in model.non_trainable_weights])
    print(f"\n{'─'*80}")
    print(f"Total parameters: {model.count_params():,}")
    print(f"  - Trainable: {trainable:,}")
    print(f"  - Non-trainable: {non_trainable:,}")
    print(f"{'='*80}")



In [8]:

# ==============================================================================
# SECTION 3: TRAINING FUNCTIONS
# ==============================================================================

def train_model(model, X_train, y_train, X_val, y_val, epochs=100, batch_size=16):
    """
    Train a model with callbacks and class weighting
    
    Parameters:
    -----------
    model : keras.Model
        Model to train
    X_train, y_train : ndarray
        Training data and labels
    X_val, y_val : ndarray
        Validation data and labels
    epochs : int
        Maximum number of epochs
    batch_size : int
        Batch size for training
        
    Returns:
    --------
    history : keras.History
        Training history object
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: {model.name}")
    print(f"{'='*80}")
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )
    
    # Callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    )
    
    # Calculate class weights for imbalanced data
    class_counts = np.bincount(y_train.astype(int))
    total = len(y_train)
    class_weight = {
        0: total / (2.0 * class_counts[0]),
        1: total / (2.0 * class_counts[1])
    }
    
    print(f"\\nTraining Configuration:")
    print(f"  Training samples: {len(X_train)}")
    print(f"  Validation samples: {len(X_val)}")
    print(f"  Batch size: {batch_size}")
    print(f"  Max epochs: {epochs}")
    print(f"  Class weights: {{0: {class_weight[0]:.3f}, 1: {class_weight[1]:.3f}}}")
    print(f"\\nCallbacks:")
    print(f"  ✓ Early stopping (patience=15)")
    print(f"  ✓ Learning rate reduction (patience=7)")
    print(f"\n{'─'*80}")
    print("Starting training...\n")
    
    # Train
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    
    print(f"\\n✓ Training complete for {model.name}!")
    print(f"  Final epoch: {len(history.history['loss'])}")
    print(f"  Best val_loss: {min(history.history['val_loss']):.4f}")
    print(f"  Best val_accuracy: {max(history.history['val_accuracy']):.4f}")
    
    return history



In [9]:

# ==============================================================================
# SECTION 4: VISUALIZATION FUNCTIONS
# ==============================================================================

def plot_training_history(histories, model_names, save_path='training_history.png'):
    """
    Plot training history for multiple models
    
    Creates a 2x2 grid showing:
    - Loss (train & val)
    - Accuracy (train & val)
    - AUC (train & val)
    - Precision (train & val)
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    metrics = ['loss', 'accuracy', 'auc', 'precision']
    titles = ['Loss', 'Accuracy', 'AUC', 'Precision']
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx // 2, idx % 2]
        
        for hist_idx, (history, model_name) in enumerate(zip(histories, model_names)):
            epochs = range(1, len(history.history[metric]) + 1)
            
            # Training
            ax.plot(epochs, history.history[metric], 
                   label=f'{model_name} (Train)', 
                   linewidth=2.5, color=colors[hist_idx], alpha=0.8)
            
            # Validation
            ax.plot(epochs, history.history[f'val_{metric}'], 
                   label=f'{model_name} (Val)', 
                   linestyle='--', linewidth=2.5, color=colors[hist_idx], alpha=0.6)
        
        ax.set_xlabel('Epoch', fontsize=13, fontweight='bold')
        ax.set_ylabel(title, fontsize=13, fontweight='bold')
        ax.set_title(f'{title} Comparison', fontsize=15, fontweight='bold', pad=15)
        ax.legend(loc='best', fontsize=10, framealpha=0.9)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    plt.suptitle('Training History - All Models (Temporal Split)', fontsize=18, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\\n✓ Training history plot saved: {save_path}")
    plt.close()


def plot_confusion_matrices(y_true_list, y_pred_list, model_names, 
                            save_path='confusion_matrices.png'):
    """
    Plot beautiful confusion matrices for multiple models side-by-side
    """
    n_models = len(model_names)
    fig, axes = plt.subplots(1, n_models, figsize=(6.5*n_models, 5.5))
    
    if n_models == 1:
        axes = [axes]
    
    for idx, (y_true, y_pred, model_name) in enumerate(zip(y_true_list, y_pred_list, model_names)):
        cm = confusion_matrix(y_true, y_pred)
        
        # Calculate percentages
        cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
        
        # Create annotations with counts and percentages
        annotations = np.empty_like(cm, dtype=object)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                count = cm[i, j]
                pct = cm_percent[i, j]
                annotations[i, j] = f'{count}\n({pct:.1f}%)'
        
        # Plot heatmap
        sns.heatmap(cm, annot=annotations, fmt='', cmap='Blues', 
                   cbar=True, ax=axes[idx], square=True, cbar_kws={'shrink': 0.8},
                   xticklabels=['No Stings (0)', 'Stings (1)'],
                   yticklabels=['No Stings (0)', 'Stings (1)'],
                   linewidths=2, linecolor='white',
                   annot_kws={'fontsize': 14, 'fontweight': 'bold'})
        
        axes[idx].set_title(f'{model_name}', fontsize=15, fontweight='bold', pad=15)
        axes[idx].set_ylabel('True Label', fontsize=13, fontweight='bold')
        axes[idx].set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
        
        # Add metrics below
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        
        metrics_text = f'Accuracy: {acc:.3f}  |  F1-Score: {f1:.3f}'
        axes[idx].text(0.5, -0.18, metrics_text, 
                      transform=axes[idx].transAxes, ha='center',
                      fontsize=12, fontweight='bold',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.3))
    
    plt.suptitle('Confusion Matrices - Test Set (Most Recent Data)', 
                fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Confusion matrices saved: {save_path}")
    plt.close()


In [10]:


def plot_model_comparison_bar(comparison_df, save_path='model_comparison_chart.png'):
    """
    Create a bar chart comparing model metrics
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    models = comparison_df['Model'].values
    test_acc = comparison_df['Test Acc'].str.replace(',', '').astype(float).values
    test_f1 = comparison_df['Test F1'].str.replace(',', '').astype(float).values
    params = comparison_df['Parameters'].str.replace(',', '').astype(int).values / 1000
    
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    # Plot 1: Accuracy and F1-Score
    ax = axes[0]
    x = np.arange(len(models))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, test_acc, width, label='Accuracy', color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    bars2 = ax.bar(x + width/2, test_f1, width, label='F1-Score', color=colors, alpha=0.5, edgecolor='black', linewidth=1.5)
    
    ax.set_ylabel('Score', fontsize=13, fontweight='bold')
    ax.set_title('Test Set Performance (Most Recent Data)', fontsize=15, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1.0])
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Plot 2: Parameter count
    ax = axes[1]
    bars = ax.bar(models, params, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    ax.set_ylabel('Parameters (thousands)', fontsize=13, fontweight='bold')
    ax.set_title('Model Complexity', fontsize=15, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, param in zip(bars, params):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
               f'{param:.1f}K', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Model comparison chart saved: {save_path}")
    plt.close()


In [11]:


# ==============================================================================
# SECTION 5: EVALUATION AND COMPARISON
# ==============================================================================

def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate a single model and print detailed metrics
    """
    print(f"\n{'─'*80}")
    print(f"EVALUATING: {model_name}")
    print(f"{'─'*80}")
    
    # Predictions
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"\\nTest Set Performance:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    
    print(f"\\nClassification Report:")
    print(classification_report(y_test, y_pred, 
                               target_names=['No Stings (0)', 'Stings (1)'],
                               digits=4))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"Confusion Matrix Breakdown:")
    print(f"  True Negatives (TN):  {tn:4d} - Correctly predicted no stings")
    print(f"  False Positives (FP): {fp:4d} - False alarms")
    print(f"  False Negatives (FN): {fn:4d} - Missed stings (CRITICAL)")
    print(f"  True Positives (TP):  {tp:4d} - Correctly predicted stings")
    
    return y_pred


def create_comparison_table(models, histories, y_true_list, y_pred_list, model_names):
    """
    Create comprehensive comparison table
    """
    results = []
    
    for model, history, y_true, y_pred, model_name in zip(
            models, histories, y_true_list, y_pred_list, model_names):
        
        # Calculate metrics
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        
        # Get final training metrics
        final_train_acc = history.history['accuracy'][-1]
        final_val_acc = history.history['val_accuracy'][-1]
        final_train_loss = history.history['loss'][-1]
        final_val_loss = history.history['val_loss'][-1]
        
        # Get best validation metrics
        best_val_acc = max(history.history['val_accuracy'])
        best_epoch = np.argmax(history.history['val_accuracy']) + 1
        
        results.append({
            'Model': model_name,
            'Parameters': f"{model.count_params():,}",
            'Best Val Acc': f"{best_val_acc:.4f}",
            'Best Epoch': best_epoch,
            'Test Acc': f"{acc:.4f}",
            'Test F1': f"{f1:.4f}",
            'Final Train Loss': f"{final_train_loss:.4f}",
            'Final Val Loss': f"{final_val_loss:.4f}",
            'Epochs Trained': len(history.history['loss'])
        })
    
    df = pd.DataFrame(results)
    
    print(f"\n{'='*80}")
    print("MODEL COMPARISON TABLE")
    print(f"{'='*80}")
    print(df.to_string(index=False))
    print(f"{'='*80}\n")
    
    return df


In [12]:


# ==============================================================================
# SECTION 6: MAIN TRAINING PIPELINE
# ==============================================================================

def main_training_pipeline(goldcoast_path, newcastle_path=None, 
                           combine_data=False, epochs=100, batch_size=16,
                           output_dir='outputs'):
    """
    Main pipeline to train and evaluate all three models using TEMPORAL SPLIT
    
    Parameters:
    -----------
    goldcoast_path : str or Path
        Path to Gold Coast NPZ file
    newcastle_path : str or Path, optional
        Path to Newcastle NPZ file
    combine_data : bool
        If True and both datasets provided, combine them for training
    epochs : int
        Maximum number of training epochs
    batch_size : int
        Batch size for training
    output_dir : str
        Directory to save outputs
        
    Returns:
    --------
    models : list
        List of trained models
    histories : list
        List of training histories
    comparison_df : DataFrame
        Comparison table
    """
    
    print("\n" + "="*80)
    print("3D CNN TRAINING PIPELINE - TEMPORAL SPLIT (NO RANDOMIZATION)")
    print("="*80)
    
    # Create output directory
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    print(f"\\nOutput directory: {output_path.absolute()}")
    
    # =====================================================
    # STEP 1: LOAD DATA
    # =====================================================
    
    # Load Gold Coast data
    X_gc, y_gc, dates_gc = load_and_preprocess_data(goldcoast_path, 'Gold Coast')
    
    # Load Newcastle data if provided
    if newcastle_path is not None:
        X_nc, y_nc, dates_nc = load_and_preprocess_data(newcastle_path, 'Newcastle')
        
        if combine_data:
            print(f"\n{'='*80}")
            print("COMBINING DATASETS")
            print(f"{'='*80}")
            
            # Pad Newcastle (12x12) to match Gold Coast (15x15)
            n_samples, n_days, n_channels, h_nc, w_nc = X_nc.shape
            h_gc, w_gc = X_gc.shape[-2:]
            
            # Calculate padding
            pad_h = (h_gc - h_nc) // 2
            pad_w = (w_gc - w_nc) // 2
            
            X_nc_padded = np.pad(X_nc, 
                                ((0, 0), (0, 0), (0, 0), 
                                 (pad_h, h_gc - h_nc - pad_h),
                                 (pad_w, w_gc - w_nc - pad_w)),
                                mode='constant', constant_values=0)
            
            print(f"✓ Newcastle padded from {X_nc.shape} to {X_nc_padded.shape}")
            
            # Combine datasets
            X_combined = np.concatenate([X_gc, X_nc_padded], axis=0)
            y_combined = np.concatenate([y_gc, y_nc], axis=0)
            dates_combined = np.concatenate([dates_gc, dates_nc], axis=0)
            
            print(f"✓ Combined shape: {X_combined.shape}")
            print(f"✓ Combined targets: {y_combined.shape}")
            
            X_data, y_data, dates_data = X_combined, y_combined, dates_combined
        else:
            print("\\n⚠ Newcastle data loaded but not combined (combine_data=False)")
            print("  Using Gold Coast data only")
            X_data, y_data, dates_data = X_gc, y_gc, dates_gc
    else:
        X_data, y_data, dates_data = X_gc, y_gc, dates_gc
    
    # =====================================================
    # STEP 2: TEMPORAL SPLIT (NO RANDOMIZATION)
    # =====================================================
    
    print(f"\n{'='*80}")
    print("TEMPORAL DATA SPLIT (CHRONOLOGICAL ORDER PRESERVED)")
    print(f"{'='*80}")
    print("⚠ IMPORTANT: Data is NOT randomized - split by time")
    print("  Training: Earliest data")
    print("  Validation: Middle period")
    print("  Test: Most recent data\n")
    
    X_train, X_val, X_test, y_train, y_val, y_test, dates_train, dates_val, dates_test = \
        temporal_train_val_test_split(X_data, y_data, dates_data, 
                                     train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)
    
    print(f"Data split summary:")
    print(f"  Training set:   {X_train.shape[0]:4d} samples ({X_train.shape[0]/len(X_data)*100:.1f}%)")
    print(f"    Date range: {dates_train[0]} to {dates_train[-1]}")
    print(f"    Class 0: {(y_train==0).sum()}, Class 1: {(y_train==1).sum()}")
    print(f"\n  Validation set: {X_val.shape[0]:4d} samples ({X_val.shape[0]/len(X_data)*100:.1f}%)")
    print(f"    Date range: {dates_val[0]} to {dates_val[-1]}")
    print(f"    Class 0: {(y_val==0).sum()}, Class 1: {(y_val==1).sum()}")
    print(f"\n  Test set:       {X_test.shape[0]:4d} samples ({X_test.shape[0]/len(X_data)*100:.1f}%)")
    print(f"    Date range: {dates_test[0]} to {dates_test[-1]}")
    print(f"    Class 0: {(y_test==0).sum()}, Class 1: {(y_test==1).sum()}")
    print(f"\n  Total:          {len(X_data):4d} samples")
    
    # Define input shape
    input_shape = X_train.shape[1:]  # (n_days, channels, H, W)
    print(f"\\nInput shape for models: {input_shape}")
    print(f"  Time steps: {input_shape[0]}")
    print(f"  Channels: {input_shape[1]}")
    print(f"  Spatial: {input_shape[2]}x{input_shape[3]}")
    
    # =====================================================
    # STEP 3: CREATE MODELS
    # =====================================================
    
    print(f"\n{'='*80}")
    print("CREATING MODELS")
    print(f"{'='*80}")
    
    model_1 = create_model_1_simple_3dcnn(input_shape, "Simple3DCNN")
    model_2 = create_model_2_c3d_lite(input_shape, "C3D_Lite")
    model_3 = create_model_3_separable_3dcnn(input_shape, "Separable3DCNN")
    
    models = [model_1, model_2, model_3]
    model_names = [m.name for m in models]
    
    for model in models:
        print_model_summary(model)
    
    # =====================================================
    # STEP 4: TRAIN MODELS
    # =====================================================
    
    print(f"\n{'='*80}")
    print("TRAINING ALL MODELS")
    print(f"{'='*80}")
    
    histories = []
    for model in models:
        history = train_model(model, X_train, y_train, X_val, y_val, 
                             epochs=epochs, batch_size=batch_size)
        histories.append(history)
    
    # =====================================================
    # STEP 5: EVALUATE ON TEST SET (MOST RECENT DATA)
    # =====================================================
    
    print(f"\n{'='*80}")
    print("EVALUATING ON TEST SET (MOST RECENT DATA)")
    print(f"{'='*80}")
    print(f"Test period: {dates_test[0]} to {dates_test[-1]}")
    
    y_true_list = []
    y_pred_list = []
    
    for model in models:
        y_pred = evaluate_model(model, X_test, y_test, model.name)
        y_true_list.append(y_test)
        y_pred_list.append(y_pred)
    
    # =====================================================
    # STEP 6: CREATE VISUALIZATIONS
    # =====================================================
    
    print(f"\n{'='*80}")
    print("CREATING VISUALIZATIONS")
    print(f"{'='*80}")
    
    plot_training_history(histories, model_names, 
                         save_path=output_path / 'training_history.png')
    
    plot_confusion_matrices(y_true_list, y_pred_list, model_names,
                           save_path=output_path / 'confusion_matrices.png')
    
    # =====================================================
    # STEP 7: CREATE COMPARISON TABLE
    # =====================================================
    
    comparison_df = create_comparison_table(models, histories, y_true_list, 
                                           y_pred_list, model_names)
    
    # Save comparison table
    comparison_path = output_path / 'model_comparison.csv'
    comparison_df.to_csv(comparison_path, index=False)
    print(f"✓ Comparison table saved: {comparison_path}")
    
    # Plot comparison chart
    plot_model_comparison_bar(comparison_df, 
                             save_path=output_path / 'model_comparison_chart.png')
    
    # =====================================================
    # STEP 8: SAVE MODELS
    # =====================================================
    
    print(f"\n{'='*80}")
    print("SAVING MODELS")
    print(f"{'='*80}")
    
    for model in models:
        model_path = output_path / f"{model.name}_model.keras"
        model.save(model_path)
        print(f"✓ Saved {model.name}: {model_path}")
    
    # Save split information
    split_info = {
        'train_dates': [dates_train[0], dates_train[-1]],
        'val_dates': [dates_val[0], dates_val[-1]],
        'test_dates': [dates_test[0], dates_test[-1]],
        'train_size': len(X_train),
        'val_size': len(X_val),
        'test_size': len(X_test)
    }
    
    split_info_df = pd.DataFrame([split_info])
    split_info_path = output_path / 'temporal_split_info.csv'
    split_info_df.to_csv(split_info_path, index=False)
    print(f"✓ Saved temporal split info: {split_info_path}")
    
    # =====================================================
    # FINAL SUMMARY
    # =====================================================
    
    print(f"\n{'='*80}")
    print("PIPELINE COMPLETE! 🎉")
    print(f"{'='*80}")
    
    print(f"\\n📁 Output Files:")
    print(f"  ✓ training_history.png         - Training curves for all models")
    print(f"  ✓ confusion_matrices.png       - Confusion matrices comparison")
    print(f"  ✓ model_comparison.csv         - Numerical comparison table")
    print(f"  ✓ model_comparison_chart.png   - Visual comparison bar chart")
    print(f"  ✓ temporal_split_info.csv      - Train/Val/Test date ranges")
    print(f"  ✓ Simple3DCNN_model.keras      - Saved model 1")
    print(f"  ✓ C3D_Lite_model.keras         - Saved model 2")
    print(f"  ✓ Separable3DCNN_model.keras   - Saved model 3")
    
    # Best model
    best_idx = comparison_df['Test Acc'].str.replace(',', '').astype(float).idxmax()
    best_model = comparison_df.loc[best_idx, 'Model']
    best_acc = comparison_df.loc[best_idx, 'Test Acc']
    best_f1 = comparison_df.loc[best_idx, 'Test F1']
    
    print(f"\\n🏆 Best Model: {best_model}")
    print(f"   Test Accuracy: {best_acc}")
    print(f"   Test F1-Score: {best_f1}")
    
    print(f"\\n⏰ Data Split Summary:")
    print(f"   Training:   {dates_train[0]} to {dates_train[-1]}")
    print(f"   Validation: {dates_val[0]} to {dates_val[-1]}")
    print(f"   Test:       {dates_test[0]} to {dates_test[-1]}")
    
    print(f"\n{'='*80}\n")
    
    return models, histories, comparison_df


# ==============================================================================
# SECTION 7: MAIN EXECUTION
# ==============================================================================

if __name__ == "__main__":
    """
    Main execution block - UPDATE THE PATHS BELOW TO YOUR DATA FILES
    """
    

    
    # ========================================================================
    # CONFIGURATION - UPDATE THESE PATHS TO YOUR DATA
    # ========================================================================
    
    current_dir = Path(os.getcwd()).parent
    
    # Path to your Gold Coast NPZ file
    goldcoast_npz = current_dir / 'data' / 'processed' / 'GoldCoast' / 'GoldCoast_7_days.npz'
    
    # Path to your Newcastle NPZ file (optional)
    newcastle_npz = current_dir / 'data' / 'processed' / 'Sydney_Newcastle' / 'Sydney_Newcastle_7_days.npz'
    
    # Training parameters
    EPOCHS = 100          # Maximum epochs (early stopping will likely stop earlier)
    BATCH_SIZE = 16       # Adjust based on GPU memory: 8, 16, or 32
    COMBINE_DATA = True  # True to combine Gold Coast + Newcastle
    OUTPUT_DIR = 'outputs_combined'  # Where to save results
    
    # ========================================================================
    # VALIDATE PATHS
    # ========================================================================

    
    if not goldcoast_npz.exists():
        print(f"\n❌ ERROR: Gold Coast data file not found!")
        print(f"   Expected: {goldcoast_npz}")
        print(f"\n   Please update the 'goldcoast_npz' path in the script.")
        print(f"   Make sure you've created the sequential dataset first.")
        import sys
        sys.exit(1)
    
    print(f"✓ Gold Coast data found: {goldcoast_npz.name}")
    
    # Check Newcastle (optional)
    has_newcastle = newcastle_npz.exists()
    if has_newcastle:
        print(f"✓ Newcastle data found: {newcastle_npz.name}")
    else:
        print(f"⚠ Newcastle data not found (optional): {newcastle_npz.name}")
        newcastle_npz = None
    
    print(f"\\nTraining Configuration:")
    print(f"  Max Epochs: {EPOCHS}")
    print(f"  Batch Size: {BATCH_SIZE}")
    print(f"  Combine Datasets: {COMBINE_DATA}")
    print(f"  Output Directory: {OUTPUT_DIR}")
    
    # ========================================================================
    # RUN TRAINING PIPELINE
    # ========================================================================
    

    models, histories, comparison_df = main_training_pipeline(
        goldcoast_path=goldcoast_npz,
        newcastle_path=newcastle_npz if COMBINE_DATA and has_newcastle else None,
        combine_data=COMBINE_DATA,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        output_dir=OUTPUT_DIR
    )
    


✓ Gold Coast data found: GoldCoast_7_days.npz
✓ Newcastle data found: Sydney_Newcastle_7_days.npz
\nTraining Configuration:
  Max Epochs: 100
  Batch Size: 16
  Combine Datasets: True
  Output Directory: outputs_combined

3D CNN TRAINING PIPELINE - TEMPORAL SPLIT (NO RANDOMIZATION)
\nOutput directory: c:\Users\VJ\Desktop\2025T3Group4\notebooks\outputs_combined

LOADING GOLD COAST DATA
File: c:\Users\VJ\Desktop\2025T3Group4\data\processed\GoldCoast\GoldCoast_7_days.npz
\nOriginal shape: (743, 7, 6, 15, 15)
  - Samples: 743
  - Days: 7
  - Channels: 6
  - Spatial: 15x15
\nTargets shape: (743,)
Date range: 2010-01-09 to 2024-12-29

────────────────────────────────────────────────────────────────────────────────
HANDLING NaN VALUES
────────────────────────────────────────────────────────────────────────────────
NaN count before: 790,552 (11.26%)
NaN count after: 0
✓ NaN values replaced with 0 (land mask)

────────────────────────────────────────────────────────────────────────────────
NORM

In [ ]:


print("="*80)
print("✓ UPDATED SCRIPT WITH TEMPORAL SPLIT!")
print("="*80)
print(f"\nFile: train_3dcnn_complete.py")
print(f"\n🔄 KEY CHANGE: NO RANDOMIZATION - Temporal Split")
print(f"\nData is now split chronologically:")
print(f"  • Training set:   64% (EARLIEST data)")
print(f"  • Validation set: 16% (MIDDLE period)")
print(f"  • Test set:       20% (MOST RECENT data)")
print(f"\nThis is appropriate for time series because:")
print(f"  ✓ Tests on future unseen data (realistic scenario)")
print(f"  ✓ Preserves temporal dependencies")
print(f"  ✓ No data leakage from future to past")
print(f"  ✓ Mimics real-world deployment")
print(f"\nAdditional output:")
print(f"  • temporal_split_info.csv - Contains exact date ranges for each split")
print(f"\nTo use:")
print(f"  1. Update file paths in the script")
print(f"  2. Run: python train_3dcnn_complete.py")
print("="*80)

In [7]:


print("="*80)
print("✓ COMPLETE ALL-IN-ONE SCRIPT CREATED!")
print("="*80)
print("\nFile: train_3dcnn_complete.py")
print(f"Size:  characters")
print("\nThis single file contains:")
print("  ✓ Data loading and NaN handling")
print("  ✓ Channel-wise normalization")
print("  ✓ Three 3D CNN architectures (Simple, C3D-Lite, Separable)")
print("  ✓ Training with early stopping and LR scheduling")
print("  ✓ Comprehensive evaluation and metrics")
print("  ✓ Multiple visualization functions")
print("  ✓ Confusion matrices with pretty formatting")
print("  ✓ Model comparison tables and charts")
print("  ✓ Individual detailed training curves")
print("  ✓ Automatic model saving")
print("  ✓ Complete error handling")
print("\n" + "="*80)
print("HOW TO USE:")
print("="*80)
print("\n1. Update the paths at the bottom of the file:")
print("   goldcoast_npz = Path('your/path/to/GoldCoast_7_days.npz')")
print("\n2. Adjust training parameters if needed:")
print("   EPOCHS = 100")
print("   BATCH_SIZE = 16")
print("   COMBINE_DATA = False")
print("\n3. Run the script:")
print("   python train_3dcnn_complete.py")
print("\n4. Wait for training to complete (~30-60 min depending on hardware)")
print("\n5. Check the generated outputs:")
print("   - training_history.png")
print("   - confusion_matrices.png")
print("   - model_comparison.csv")
print("   - *_model.keras (saved models)")
print("   - plots/ folder with detailed analysis")
print("\n" + "="*80)
print("✅ Ready to train! Good luck with your project! 🚀")
print("="*80 + "\n")

✓ COMPLETE ALL-IN-ONE SCRIPT CREATED!

File: train_3dcnn_complete.py
Size:  characters

This single file contains:
  ✓ Data loading and NaN handling
  ✓ Channel-wise normalization
  ✓ Three 3D CNN architectures (Simple, C3D-Lite, Separable)
  ✓ Training with early stopping and LR scheduling
  ✓ Comprehensive evaluation and metrics
  ✓ Multiple visualization functions
  ✓ Confusion matrices with pretty formatting
  ✓ Model comparison tables and charts
  ✓ Individual detailed training curves
  ✓ Automatic model saving
  ✓ Complete error handling

HOW TO USE:

1. Update the paths at the bottom of the file:
   goldcoast_npz = Path('your/path/to/GoldCoast_7_days.npz')

2. Adjust training parameters if needed:
   EPOCHS = 100
   BATCH_SIZE = 16
   COMBINE_DATA = False

3. Run the script:
   python train_3dcnn_complete.py

4. Wait for training to complete (~30-60 min depending on hardware)

5. Check the generated outputs:
   - training_history.png
   - confusion_matrices.png
   - model_compar